# 14. 공간과 공연노출 독립감사

2026-10-05 감사. 종로구·마포구 151개 상권, 2023~2025년 12개 분기의 공연 노출을 점검한다. 기존 03·04·12번의 연결/계산 검증을 반복해 결함으로 취급하지 않고, **실제 측정 대상·공간 중첩·자료 완전성**의 추가 쟁점을 수치로 확인한다.

검증된 20261004_154020_1c7d3b 스냅샷과 기존 분석 산출물만 읽는다. 네트워크 요청·수집 중 CSV 읽기·기존 파이프라인 변경은 하지 않는다. 결과는 `reports/20261005/audit_20261005/exposure/`에만 저장한다. 달력상 기간의 일수는 실제 상연일수·관객수가 아니며 대안 노출로 확정하지 않는다.

In [ ]:
# 현재 구조 및 과거 고정 명세의 경로를 함께 해석합니다.
from pathlib import Path as _ProjectPath
import json as _bootstrap_json
_project_home = next(p for p in [_ProjectPath.cwd(), *_ProjectPath.cwd().parents]
                     if (p / 'environment/project_paths.ipynb').is_file())
_project_notebook = _bootstrap_json.loads((_project_home / 'environment/project_paths.ipynb').read_text(encoding='utf-8'))
for _project_cell in _project_notebook['cells']:
    if 'project-runtime' in _project_cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_project_cell['source']), 'environment/project_paths.ipynb', 'exec'), globals())

from pathlib import Path
import json, hashlib, itertools
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
ROOT = PROJECT_ROOT
OUT = project_path('reports/20261005/audit_20261005/exposure')
OUT.mkdir(parents=True, exist_ok=True)
RUN = '20261004_154020_1c7d3b'
manifest = project_json_loads((project_path(f'reports/kopis_collection/{RUN}/snapshot_manifest.json')).read_text(encoding='utf-8'))
assert manifest['validated'] and manifest['run_id'] == RUN
inputs = {}
def read_csv(rel, **kwargs):
    path = project_path(rel)
    inputs[rel] = hashlib.sha256(path.read_bytes()).hexdigest()
    return pd.read_csv(path, **kwargs)
def save(frame, name):
    frame.to_csv(OUT / (name + '.csv'), index=False, encoding='utf-8-sig')
for record in [*manifest['input_sources'], *manifest['files'].values()]:
    assert hashlib.sha256((project_path(record['path'])).read_bytes()).hexdigest() == record['sha256']
link = read_csv('dataset/analysis/performance_facility_linkage.csv', dtype={'mt20id':str,'mt10id':str})
active = read_csv('dataset/analysis/performance_facility_quarter.csv', dtype={'quarter':str})
distance = read_csv('dataset/analysis/venue_area_distances_800m.csv', dtype={'area_id':str})
areas = read_csv('dataset/analysis/study_areas.csv', dtype={'area_id':str})
aqp = read_csv('dataset/analysis/area_performance_quarter.csv', dtype={'quarter':str,'area_id':str})
fac = read_csv(manifest['files']['facility_details']['path'], dtype={'mt10id':str})
detail = read_csv(manifest['files']['performance_details']['path'])
months = read_csv('dataset/processed/kopis/performance_month_observations_seoul_2023_2025.csv', dtype={'query_month':str})
for c in ['start_date','end_date']: link[c] = pd.to_datetime(link[c])
near500 = distance.loc[distance.distance_m.le(500)].copy()
near800_ids = set(distance.mt10id)
local = link[link.mt10id.isin(near800_ids)].copy()
assert local.link_method.eq('official_detail').all()
display(pd.DataFrame({'범위':['서울 목록','상세 스냅샷','800m 내 실제 이용 공연','상권','분기'],
                    '개수':[len(link),len(detail),len(local),len(areas),aqp.quarter.nunique()]}))


## 1. 36개월 존재와 기간 검색의 실제 동작

관측된 공연의 시작·종료일로 예상한 모든 공연×월 조합을 생성하고, 원본 XML 및 가공 월별 관측쌍과 비교한다. 특히 2023년 이전 시작, 2025년 이후 종료된 공연의 포함 여부를 검사한다. 이 검사는 **이미 관측된 ID의 기간 누락**을 확인하며 애초 목록에 없는 공연의 존재를 증명하거나 부정하지는 못한다.

[KOPIS 공식 Open API 가이드](https://kopis.or.kr/upload/openApi/공연예술통합전산망OpenAPI개발가이드.pdf), 2026-04-23 v5.0, 1~4쪽: 기간 검색 인자와 공연·시설·공연장 ID, 공연기간과 시간 안내를 구분한다. 인자 이름만 보고 ‘시작일이 검색 월 안에 있는 공연만 반환한다’고 단정하지 않고 실제 원문으로 확인한다.

In [ ]:
raw_pairs, raw_counts, raw_paths = set(), [], []
for folder in sorted((project_path('dataset/raw/kopis/performance_lists')).iterdir()):
    if not folder.is_dir(): continue
    for path in sorted(folder.glob('*.xml')):
        blob = path.read_bytes()
        inputs[path.relative_to(ROOT).as_posix()] = hashlib.sha256(blob).hexdigest()
        records = ET.fromstring(blob).findall('.//db')
        raw_counts.append({'month':folder.name,'page':int(path.stem),'rows':len(records)})
        raw_pairs.update((r.findtext('mt20id'), folder.name) for r in records)
observed_pairs = set(map(tuple, months[['mt20id','query_month']].values))
expected_pairs = set()
for m in pd.period_range('2023-01','2025-12',freq='M'):
    selected = link.loc[link.start_date.le(m.end_time) & link.end_date.ge(m.start_time),'mt20id']
    expected_pairs.update((pid,m.strftime('%Y%m')) for pid in selected)
missing_pairs = expected_pairs - observed_pairs
extra_pairs = observed_pairs - expected_pairs
cross_window = link[link.start_date.lt('2023-01-01') & link.end_date.gt('2025-12-31')].copy()
cross_window['observed_months'] = cross_window.mt20id.map(months.groupby('mt20id').query_month.nunique())
month_audit = pd.DataFrame([
 {'check':'raw_processed_month_pair_difference','count':len(raw_pairs.symmetric_difference(observed_pairs))},
 {'check':'expected_overlap_pairs','count':len(expected_pairs)},
 {'check':'observed_pairs','count':len(observed_pairs)},
 {'check':'missing_expected_overlap_pairs','count':len(missing_pairs)},
 {'check':'extra_outside_period_pairs','count':len(extra_pairs)},
 {'check':'pre2023_start_ids','count':int(link.start_date.lt('2023-01-01').sum())},
 {'check':'pre2023_start_post2025_end_ids','count':len(cross_window)},
 {'check':'cross_window_ids_observed_36_months','count':int(cross_window.observed_months.eq(36).sum())}])
save(month_audit,'monthly_overlap_audit')
save(cross_window[['mt20id','prfnm','start_date','end_date','observed_months']],'cross_window_performances')
save(pd.DataFrame(sorted(missing_pairs),columns=['mt20id','query_month']),'missing_month_pairs')
display(month_audit)
display(cross_window[['mt20id','prfnm','start_date','end_date','observed_months']].head(6))


**해석:** 월별 XML 원문과 기간 중복 로직의 일치를 아래 결과로 판정한다. 시작일 검색 때문에 과거 시작 장기공연이 모두 빠졌다는 주장은 실제 원문의 장기공연 포함 사례와 양립하지 않는다. 전체 모집단의 완전성은 별도의 외부 총량과 대조하지 않았으므로 미검증으로 남긴다.

## 2. 공연 ID 수와 공연량은 어느 정도 다른가

800m 이내 실제 연결된 공연에 대해 연구기간 안의 달력상 기간 길이를 계산한다. 500m 상권×분기에는 동일 ID 1회 집계와 달력상 기간 합계의 상관 및 변동을 비교한다. 휴연·실제 회차·좌석·관객수는 관측하지 않았으므로 기간 합계를 실제 공연 강도로 해석하지 않는다.

In [ ]:
local['window_days'] = (local.end_date.clip(upper=pd.Timestamp('2025-12-31')) - local.start_date.clip(lower=pd.Timestamp('2023-01-01'))).dt.days + 1
local['duration_group'] = pd.cut(local.window_days,bins=[0,1,7,30,90,1096], labels=['1일','2~7일','8~30일','31~90일','91일 이상'])
duration = local.groupby('duration_group',observed=False).agg(performance_ids=('mt20id','nunique'),envelope_days=('window_days','sum')).reset_index()
duration['id_share_pct'] = duration.performance_ids / len(local)*100
duration['envelope_share_pct'] = duration.envelope_days / local.window_days.sum()*100
genres = local.groupby('genrenm').agg(performance_ids=('mt20id','nunique'),median_envelope_days=('window_days','median'),envelope_days=('window_days','sum')).reset_index()
save(duration,'duration_mix');save(genres,'genre_mix')
act = active[active.mt20id.isin(local.mt20id)].merge(local[['mt20id','start_date','end_date','genrenm']],on='mt20id',validate='many_to_one')
qbounds={f'{p.year}{p.quarter}':(p.start_time.normalize(),p.end_time.normalize()) for p in pd.period_range('2023Q1','2025Q4',freq='Q')}
act['overlap_days'] = [(min(e,qbounds[q][1])-max(s,qbounds[q][0])).days+1 for s,e,q in zip(act.start_date,act.end_date,act.quarter)]
exposures=act.merge(near500,on='mt10id',validate='many_to_many')
alt=exposures.groupby(['area_id','quarter']).agg(id_count=('mt20id','nunique'),envelope_days=('overlap_days','sum')).reset_index()
alt=aqp[['area_id','quarter','known_performance_count_500m']].merge(alt,on=['area_id','quarter'],how='left',validate='one_to_one').fillna({'id_count':0,'envelope_days':0})
assert alt.id_count.eq(alt.known_performance_count_500m).all()
alt=alt.merge(areas[['area_id','district']],on='area_id',validate='many_to_one')
for col in ['id_count','envelope_days']:
    alt['log_'+col] = np.log1p(alt[col])
    # 완전한 상권×분기 격자에서 상권·자치구×분기 효과를 제거한다.
    alt['within_'+col] = alt['log_'+col]-alt.groupby('area_id')['log_'+col].transform('mean')-alt.groupby(['district','quarter'])['log_'+col].transform('mean')+alt.groupby('district')['log_'+col].transform('mean')
metric_corr=pd.DataFrame([{'scale':'log1p_levels','correlation':alt.log_id_count.corr(alt.log_envelope_days)},
 {'scale':'area_and_district_quarter_residuals','correlation':alt.within_id_count.corr(alt.within_envelope_days)}])
save(alt,'count_vs_envelope_days');save(metric_corr,'count_vs_envelope_correlation')
display(duration);display(genres.sort_values('performance_ids',ascending=False));display(metric_corr)


**해석:** 동일한 1개 ID라도 단일 날짜 공연과 장기 공연은 기간 길이와 가능한 관객 유입이 다르다. 두 지표의 상관이 높아도 실제 회차를 검증한 것은 아니다. 분기 내 달력상 기간 합계는 등록 ID 구성에 대한 민감도용일 뿐, 주효과 추정의 대체 정답이 아니다.

## 3. 상권 폴리곤의 크기와 중첩

현재 경계에서 면적, 500m 확장 범위, 상권 간 면적 중첩을 계산한다. 폴리곤까지 거리가 0m라는 것은 시설점이 상권 내부에 있다는 뜻이다. 상권의 모든 음식점에서 500m 이내이거나 실제 보행거리가 500m 이내라는 뜻은 아니다. 경계를 임의 변경하지 않는다.

In [ ]:
import shapefile, shapely
from shapely.geometry import shape, Point
from pyproj import CRS, Transformer
source=project_json_loads((project_path('reports/20261005/linkage_analysis_summary.json')).read_text(encoding='utf-8'))
boundary=next(r for r in source['source_files'] if r['role']=='boundary_shp')
for r in source['source_files']:
    if r['role'].startswith('boundary_'):
        assert hashlib.sha256((project_path(r['path'])).read_bytes()).hexdigest()==r['sha256']
        inputs[r['path']]=r['sha256']
shp=project_path(boundary['path'])
reader=shapefile.Reader(str(shp),encoding=shp.with_suffix('.cpg').read_text().strip())
assert CRS.from_wkt(shp.with_suffix('.prj').read_text()).to_epsg()==5181
geometries={}; geo_rows=[]
for item in reader.iterShapeRecords():
    rec=item.record.as_dict(); aid=str(rec['TRDAR_CD']).removesuffix('.0')
    if aid not in set(areas.area_id): continue
    g=shapely.make_valid(shape(item.shape.__geo_interface__))
    if g.geom_type=='GeometryCollection':g=shapely.union_all([v for v in g.geoms if v.geom_type in ['Polygon','MultiPolygon']])
    geometries[aid]=g
    geo_rows.append({'area_id':aid,'area_type':rec.get('TRDAR_SE_C',rec.get('TRDAR_SE_1','unknown')),
      'area_m2':g.area,'buffer500_m2':g.buffer(500).area,'max_bbox_extent_m':max(g.bounds[2]-g.bounds[0],g.bounds[3]-g.bounds[1])})
geo=pd.DataFrame(geo_rows).merge(areas,on='area_id',validate='one_to_one')
overlap_rows=[]
for a,b in itertools.combinations(sorted(geometries),2):
    g,h=geometries[a],geometries[b]
    intersection=g.intersection(h).area
    if intersection>1:
        overlap_rows.append({'area_a':a,'area_b':b,'overlap_m2':intersection,'share_of_smaller':intersection/min(g.area,h.area)})
polygon_overlap=pd.DataFrame(overlap_rows)
geo['zero_distance_facilities']=geo.area_id.map(distance[distance.distance_m.eq(0)].groupby('area_id').mt10id.nunique()).fillna(0).astype(int)
geo['near500_facilities']=geo.area_id.map(near500.groupby('area_id').mt10id.nunique()).fillna(0).astype(int)
geo['performance_ids500']=geo.area_id.map(exposures.groupby('area_id').mt20id.nunique()).fillna(0).astype(int)
save(geo,'area_geometry_and_exposure');save(polygon_overlap,'polygon_overlaps')
display(geo.sort_values('area_m2',ascending=False)[['area_id','area_name','area_type','area_m2','buffer500_m2','max_bbox_extent_m','zero_distance_facilities','performance_ids500']].head(10))
display(geo[['area_m2','buffer500_m2','max_bbox_extent_m']].describe(percentiles=[.1,.5,.9]))
print('1㎡ 초과 중첩 상권쌍:',len(polygon_overlap))


In [ ]:
zero=exposures[exposures.distance_m.eq(0)]
zero_metrics=pd.DataFrame([{'metric':'area_venue_links_500m','value':len(near500)},
 {'metric':'zero_distance_area_venue_links','value':int(near500.distance_m.eq(0).sum())},
 {'metric':'zero_distance_unique_facilities','value':zero.mt10id.nunique()},
 {'metric':'area_performance_quarter_records_500m','value':len(exposures)},
 {'metric':'zero_distance_performance_quarter_records','value':len(zero)},
 {'metric':'zero_distance_record_share_pct','value':len(zero)/len(exposures)*100}])
transformer=Transformer.from_crs(4326,5181,always_xy=True)
x,y=transformer.transform(fac.lo.to_numpy(),fac.la.to_numpy())
points=dict(zip(fac.mt10id,shapely.points(x,y)))
near500['centroid_distance_m']=[points[f].distance(geometries[a].centroid) for a,f in zip(near500.area_id,near500.mt10id)]
near500['centroid_gt500m']=near500.centroid_distance_m.gt(500)
centroid_effect=exposures.merge(near500[['area_id','mt10id','centroid_gt500m']],on=['area_id','mt10id'],validate='many_to_one')
centroid_summary=pd.DataFrame([{'metric':'polygon500_links_centroid_gt500','value':int(near500.centroid_gt500m.sum())},
 {'metric':'polygon500_performance_quarter_records_centroid_gt500','value':int(centroid_effect.centroid_gt500m.sum())},
 {'metric':'polygon500_records_centroid_gt500_pct','value':centroid_effect.centroid_gt500m.mean()*100}])
save(zero_metrics,'zero_distance_metrics');save(centroid_summary,'distance_definition_sensitivity');save(near500,'polygon_vs_centroid_links')
display(zero_metrics);display(centroid_summary)


**해석:** 상권 유형·면적에 따라 같은 500m 기준이라도 조사 공간의 크기가 다르다. 중심점 거리 비교는 정의의 민감도를 보여주기 위한 진단이며, 중심점이 음식점 위치보다 더 정확하다는 근거는 아니다. 매출의 관측 단위가 중첩 상권인 경우 합계를 서로 배타적인 시장으로 해석하지 않는다. 현재 경계의 과거 유효성은 기존 문서가 이미 인정한 한계이고 이번 수치로 해결되지 않는다.

## 4. 같은 공연을 공유하는 상권과 군집 독립성

500m 안의 실제 공연을 기준으로 상권별 ID 집합을 만들고 교집합·Jaccard 비율을 계산한다. 동일 공연×분기가 몇 상권에 배분되는지, 인접 상권이 노출 시계열을 얼마나 공유하는지도 측정한다. 이것은 상권 간 잔차 상관을 직접 검정한 결과가 아니라 상권 군집 간 독립성을 별도로 검토해야 하는 공간 구조의 증거다.

In [ ]:
multiplicity=exposures.groupby(['mt20id','quarter']).area_id.nunique().rename('area_count').reset_index()
performance_sets={aid:set(g.mt20id) for aid,g in exposures.groupby('area_id')}
shared_rows=[]
for a,b in itertools.combinations(sorted(performance_sets),2):
    sa,sb=performance_sets[a],performance_sets[b]; shared=len(sa&sb)
    if shared:
        shared_rows.append({'area_a':a,'area_b':b,'shared_ids':shared,'jaccard':shared/len(sa|sb),'fraction_smaller':shared/min(len(sa),len(sb))})
shared=pd.DataFrame(shared_rows)
name=areas.set_index('area_id').area_name
for c in ['a','b']:shared['area_name_'+c]=shared['area_'+c].map(name)
profiles=aqp.pivot(index='area_id',columns='quarter',values='known_performance_count_500m')
profile_corr=profiles.T.corr()
shared['exposure_profile_correlation']=[profile_corr.loc[a,b] for a,b in zip(shared.area_a,shared.area_b)]
shared['identical_profile']=[profiles.loc[a].equals(profiles.loc[b]) for a,b in zip(shared.area_a,shared.area_b)]
share_summary=pd.DataFrame([
 {'metric':'unique_performance_quarter_pairs_500m','value':len(multiplicity)},
 {'metric':'mean_areas_per_performance_quarter','value':multiplicity.area_count.mean()},
 {'metric':'median_areas_per_performance_quarter','value':multiplicity.area_count.median()},
 {'metric':'maximum_areas_per_performance_quarter','value':multiplicity.area_count.max()},
 {'metric':'pairs_shared_by_multiple_areas_pct','value':multiplicity.area_count.gt(1).mean()*100},
 {'metric':'areas_with_observed_performance','value':len(performance_sets)},
 {'metric':'area_pairs_with_shared_performances','value':len(shared)},
 {'metric':'area_pairs_jaccard_ge_0_8','value':int(shared.jaccard.ge(.8).sum())},
 {'metric':'identical_positive_exposure_profile_pairs','value':int(shared.identical_profile.sum())}])
save(multiplicity,'performance_spatial_multiplicity');save(shared,'shared_performance_area_pairs');save(share_summary,'spatial_sharing_summary')
display(share_summary)
display(shared.sort_values(['jaccard','shared_ids'],ascending=False)[['area_name_a','area_name_b','shared_ids','jaccard','exposure_profile_correlation','identical_profile']].head(12))


**해석:** 같은 공연과 지역 충격을 공유하는 상권이 많으면 ‘상권별 군집 표준오차’가 상권 사이의 의존성까지 자동으로 처리하지 않는다. 공유 노출만으로 기존 표준오차가 반드시 과소 추정되었다고 확정할 수는 없다. 공간 또는 상권군 단위 추론 민감도와 잔차 공간상관 검정이 필요한지 검토할 근거다. 상권×업종 행 수를 독립적인 공연 관측치 수로 표현하면 안 된다.

### 보충: 반경별 공유와 연간 노출 구성

500m에서 확인된 공간 공유가 반경 선택에만 의존하는지 300·800m에서도 동일한 방법으로 집계한다. 연간 값은 해당 연도에 하루 이상 포함된 고유 ID와 그해 달력상 기간의 합계이며, 분기 ID를 단순 합산하지 않는다.

In [ ]:
radius_rows=[]
for radius in [300,500,800]:
    obs=act.merge(distance.loc[distance.distance_m.le(radius)],on='mt10id',validate='many_to_many')
    mult=obs.groupby(['mt20id','quarter']).area_id.nunique()
    radius_rows.append({'radius_m':radius,'unique_performance_ids':obs.mt20id.nunique(),
      'unique_performance_quarter_pairs':len(mult),'area_performance_quarter_records':len(obs),
      'mean_areas_per_performance_quarter':mult.mean(),'maximum_areas_per_performance_quarter':mult.max(),
      'multiple_area_share_pct':mult.gt(1).mean()*100,'zero_distance_record_share_pct':obs.distance_m.eq(0).mean()*100})
radius_summary=pd.DataFrame(radius_rows)
annual_rows=[]
for year in [2023,2024,2025]:
    start,end=pd.Timestamp(f'{year}-01-01'),pd.Timestamp(f'{year}-12-31')
    sub=local[local.start_date.le(end)&local.end_date.ge(start)].copy()
    days=(sub.end_date.clip(upper=end)-sub.start_date.clip(lower=start)).dt.days+1
    annual_rows.append({'year':year,'performance_ids800':len(sub),'envelope_days800':int(days.sum()),
      'one_day_ids':int(days.eq(1).sum()),'long90plus_ids':int(days.gt(90).sum())})
annual=pd.DataFrame(annual_rows)
annual['id_yoy_pct']=annual.performance_ids800.pct_change()*100
annual['envelope_yoy_pct']=annual.envelope_days800.pct_change()*100
save(radius_summary,'radius_sharing_summary');save(annual,'annual_count_and_envelope')
display(radius_summary);display(annual)


## 5. 좌표·시설·공연장의 시간 단위

공식 시설 ID는 같은 건물·시설을 연결할 뿐, 과거 좌표 이력이나 개별 홀/출입구를 보장하지 않는다. 현재 입력의 홀 개수, 한 시설에 여러 홀 ID가 쓰이는 정도, 공연 기간보다 뒤인 개관 연도 같은 검토 후보를 점검한다. 개관 연도 차이는 입력 의미/이전/등록 수정 중 무엇인지 원자료만으로 확정하지 않는다.

In [ ]:
local_detail=detail[detail.mt20id.isin(local.mt20id)].copy()
halls=local_detail.groupby('mt10id').agg(performance_ids=('mt20id','nunique'),hall_ids=('mt13id','nunique')).reset_index()
halls=halls.merge(fac[['mt10id','fcltynm','mt13cnt','opende','seatscale']],on='mt10id',validate='one_to_one')
local_facilities=fac[fac.mt10id.isin(local.mt10id)].copy()
local_facilities['coordinate_key']=local_facilities.la.round(7).astype(str)+','+local_facilities.lo.round(7).astype(str)
coordinate_groups=local_facilities.groupby('coordinate_key').agg(facility_count=('mt10id','nunique'),facilities=('fcltynm',lambda s:' | '.join(s))).reset_index()
opening=local[['mt20id','mt10id','prfnm','start_date']].merge(fac[['mt10id','opende','fcltynm']],on='mt10id',validate='many_to_one')
opening['opening_year']=pd.to_numeric(opening.opende,errors='coerce')
opening_candidates=opening[opening.opening_year.between(1800,2026)&opening.start_date.dt.year.lt(opening.opening_year)]
facility_metrics=pd.DataFrame([
 {'metric':'facilities_with_used_performances_800m','value':len(halls)},
 {'metric':'facilities_with_multiple_used_hall_ids','value':int(halls.hall_ids.gt(1).sum())},
 {'metric':'performance_ids_in_multiple_hall_facilities','value':int(halls.loc[halls.hall_ids.gt(1),'performance_ids'].sum())},
 {'metric':'distinct_coincident_coordinate_groups','value':int(coordinate_groups.facility_count.gt(1).sum())},
 {'metric':'performances_starting_before_recorded_opening_year','value':len(opening_candidates)}])
save(halls,'facility_hall_structure');save(coordinate_groups[coordinate_groups.facility_count.gt(1)],'coincident_facility_coordinates')
save(opening_candidates,'opening_year_review_candidates');save(facility_metrics,'facility_resolution_summary')
display(facility_metrics)
display(opening_candidates[['mt20id','prfnm','start_date','opende','fcltynm']].head(10))


**해석:** 여러 홀에서 열린 공연을 시설 대표 좌표 하나로 취급하는 것은 시설 단위 공간분석의 해상도 선택이다. 실제 음식점별 접근성으로 범위를 확장할 때 출입구/보행 경로/과거 시설 위치 확인이 필요하다. 좌표 중복·개관연도 역전 후보는 검토 목록이며 오류나 이전을 확정한 목록이 아니다.

## 6. 재현 가능한 요약과 입력 불변 확인

확인된 측정 구조, 해결된 의심, 남은 미검증 항목을 구분하여 저장한다. 기존 결과를 수정하지 않고 감사 결과를 별도 보존한다.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
font_names={f.name for f in fm.fontManager.ttflist}
plt.rcParams['font.family']='Malgun Gothic' if 'Malgun Gothic' in font_names else 'DejaVu Sans'
plt.rcParams['axes.unicode_minus']=False
fig,axes=plt.subplots(1,2,figsize=(12,4))
x=np.arange(len(duration));w=.38
axes[0].bar(x-w/2,duration.id_share_pct,w,label='공연 ID 비중')
axes[0].bar(x+w/2,duration.envelope_share_pct,w,label='달력상 기간 비중')
axes[0].set_xticks(x,duration.duration_group.astype(str));axes[0].set_ylabel('%');axes[0].legend()
axes[0].set_title('공연 ID 수와 기간 구성의 차이 (800m)')
axes[1].hist(multiplicity.area_count,bins=np.arange(0.5,multiplicity.area_count.max()+1.5),color='#557d9e')
axes[1].set_xlabel('같은 공연×분기를 공유한 상권 수');axes[1].set_ylabel('공연×분기 조합 수')
axes[1].set_title('500m 노출의 공간 공유')
fig.tight_layout();fig.savefig(OUT/'measurement_and_spatial_sharing.png',dpi=160);plt.show();plt.close(fig)


In [ ]:
def metric(frame,key):return float(frame.set_index('metric').loc[key,'value'])
summary={
 'run_id':RUN,'network_requests':0,'local800_performance_ids':len(local),'local800_official_only':bool(local.link_method.eq('official_detail').all()),
 'monthly_overlap_audit':month_audit.set_index('check')['count'].to_dict(),
 'one_day_id_share_pct':float(duration.iloc[0].id_share_pct),'one_day_envelope_share_pct':float(duration.iloc[0].envelope_share_pct),
 'count_envelope_correlations':metric_corr.set_index('scale').correlation.to_dict(),
 'geometry':{'areas':len(geo),'overlap_pairs_gt1m2':len(polygon_overlap),'min_area_m2':float(geo.area_m2.min()),'max_area_m2':float(geo.area_m2.max()),'median_area_m2':float(geo.area_m2.median()),'max_extent_m':float(geo.max_bbox_extent_m.max())},
 'zero_distance':zero_metrics.set_index('metric').value.to_dict(),
 'centroid_comparison':centroid_summary.set_index('metric').value.to_dict(),
 'spatial_sharing':share_summary.set_index('metric').value.to_dict(),
 'facility_resolution':facility_metrics.set_index('metric').value.to_dict(),
 'classification':{'confirmed_measurement_limits':['공연ID와실제상연강도불일치','폴리곤크기에따른500m범위차이','상권간공연노출공유'],
 'unverified':['누락된미관측공연전체규모','역사적시설위치및경계','상권간잔차공간상관과기존표준오차편향'],
 'not_established':['시작일검색때문에장기공연전부누락','좌표누락때문에현재800m노출손실']}}
(OUT/'audit_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
provenance=pd.DataFrame([{'path':rel,'sha256':sha,'unchanged':hashlib.sha256((project_path(rel)).read_bytes()).hexdigest()==sha} for rel,sha in inputs.items()])
assert provenance.unchanged.all()
save(provenance,'input_integrity')
checks=pd.DataFrame([{'check':'all_read_inputs_unchanged','passed':bool(provenance.unchanged.all())},
 {'check':'local800_official_ids_only','passed':bool(local.link_method.eq('official_detail').all())},
 {'check':'exposure_reconstruction_exact','passed':bool(alt.id_count.eq(alt.known_performance_count_500m).all())},
 {'check':'raw_processed_month_pairs_identical','passed':raw_pairs==observed_pairs}])
save(checks,'execution_checks');display(checks)
print('감사 저장 위치:',OUT)
